# Diffusion profiles in flexible materials

This notebook demonstrates how to compute free-energy profiles for diffusion through a flexible framework. The key idea is to account for the fact that the gate opening changes from one local structure to another, so the host is not represented by a single static potential.

The workflow is:

1. Define a guest molecule and a set of host structures spanning different gate sizes.
2. For each gate size and local structural configuration, compute an equilibrium adsorption density at a target temperature and pressure.
3. Project the density onto a collective variable (CV) that follows the diffusion coordinate through the gate.
4. Convert the projected density to a free-energy profile.
5. Either average the profiles over the different gate sizes or construct an effective average host potential to obtain a representative flexible diffusion profile.

The example intentionally uses a reduced resolution to keep the computation tractable, this leads to suboptimal results. The same procedure can be repeated with a finer grid and more states for production-quality results. This notebook is intended purely as a tutorial and a demonstration of the workflow. 


In [ ]:
# Imports

import numpy as np
import matplotlib.pyplot as plt
import h5py

from cmmdft.units_constants import *

from cmmdft.system import NanoporousHost, SphericalLJGuest, DualModelGuest, EmptyHost, GuestMixture, EffectiveAverageHost
from cmmdft.program import Program
from cmmdft.solver import Picard, Anderson, QuasiNewton
from cmmdft.calculator import Calculator
from cmmdft.plotter import Plotter
from cmmdft.eos import PCSAFTEOS, PengRobinsonMixEOS
from cmmdft.guest_dict import SAFT_pars_dict, chk_dict, pars_dict, LJ_pars_dict
from cmmdft.tools import get_chempot_key

In [ ]:
prefix = 'OutputFiles/FlexibleDiffusionPath'

# Guest definition: a C2H6 model using the dual-model representation.
# The molecular structure and force-field data define the guest interaction model
# used during the host-guest free-energy calculations.
guestname = 'C2H6_UA'
mass, sigma, epsilon = LJ_pars_dict['C2H6']

guestchk = 'InputFiles/Guest/C2H6/struct_CA.chk'
guestpar = 'InputFiles/Guest/C2H6/pars_vdw_CA_trappe.txt'
guest = DualModelGuest(guestname, mass, sigma, epsilon, struct=guestchk, par=guestpar, m=1, ffname='TRAPPE-PCSAFT', hs_def='exp')

# Force-field and functional labels identify the model combination used for the host and guest.
# The grid suffix controls the spatial resolution of the density grid in the calculation.
ff_suffix = 'AMBER_TRAPPE'
funct_suffix = 'atWBII_MFA_WDAc'

grid_suffix = '1_0' # A low resolution is chosen for the sake of efficiency in this example


In [ ]:
# Thermodynamic properties. The calculations are carried out over a small pressure range
# to illustrate the workflow and to generate profiles that can be compared across conditions.
temperature = 300*kelvin
pressures = np.linspace(0.1,50,5)*bar

# Gate sizes are generated from a range of local ZIF-8 structures. In a flexible framework,
# each structure corresponds to a different aperture and therefore a different host potential.
gate_sizes_float = np.arange(3.5750, 3.7550, 0.04)
gate_sizes = [f'{gs:0.4f}' for gs in gate_sizes_float]
selections = np.arange(5)


In [ ]:
# Calculating the equilibrium density for the different gate sizes and structural selections.
# This loop is the main computational part of the example: each host structure defines a
# different local environment, and each pressure yields a different equilibrium density.
#
# The procedure is intentionally simplified here for speed, but the workflow matches a
# production calculation.
#
# NOTE: this cell can take several minutes to complete depending on the machine.

for gate_size in gate_sizes:
    for selection in selections:
        hostname = 'ZIF8'
        
        hostchk = f'InputFiles/Host/ZIF8/ZIF8_Gates/{gate_size}/structure_{selection}.pdb'
        hostpar = 'InputFiles/Host/ZIF8/AMBER_system.xml'
        
        # A host instance is constructed separately for each local structure.
        # The microscopic gate opening changes the external potential seen by the guest.
        host = NanoporousHost(hostname, hostchk, hostpar,
                      unit_energy='kjmol', unit_distance='angstrom', unit_sigma='nanometer', unit_mass='amu') #it is important to specify the units of the input file

        suffix = f'{gate_size}/{selection}'

        # Program initialization
        program = Program(prefix=prefix, hostname=hostname, guestname=guestname, 
                            ff_suffix=ff_suffix, funct_suffix=funct_suffix, grid_suffix=grid_suffix, suffix=suffix)

        program.set_system(host, guest)
        program.set_grid(npoints=[16,16,16])
        program.set_eos('MBWR')
        program.init_free_energy(temperature)

        program.fener.add_external_potential(temperature)
        program.fener.add_hard_sphere()
        program.fener.add_mean_field()
        program.set_temperature(temperature)

        # Multiple convergence criteria are used to stabilize the iterative solution.
        criteria = ['riue', 'res']
        thresholds = [1e-4, 1e-6]

        solver = Anderson(program, criterion=criteria, threshold=thresholds)
        program.set_solver(solver)

        calc = Calculator(program)

        for P in pressures:
            program.solve(pressure=P)
            calc.save_loading_and_grand_potential(temperature=temperature, pressure=P) # Saves loading and grand potential per calculation to an npz file for later use (necessary for later integration)

        # The collective variable (CV) is chosen along the diffusion direction through the gate.
        # In ZIF-8, the relevant axis corresponds to the 111 diagonal of the unit cell.
        carbon_indices = np.array([171,191,175,187,179,183]) # indices of the atoms defining the gate opening
        step_dist = program.grid.spacings[0]/np.sqrt(3) # because the diffusion axis is the 111 diagonal for this unit cell (adjust as needed)
        distance_from_axis = 7*angstrom # necessary to keep the integration to the relevant pores (adjust as needed)
        cvs_limits = np.array([-7.5,7.5])*angstrom # defines the edges of the cv range

        cvs, cvs_mat, dist_mask = calc.collective_variable(ring_indices=carbon_indices, dist_from_axis=distance_from_axis,
                                                            step_dist=step_dist, cvs_limits=cvs_limits, supercell=True)

        for P in pressures:
            calc.project_density(cvs, cvs_mat, dist_mask, temperature=temperature, pressure=P)
            calc.free_energy_path(temperature=temperature, pressure=P)        

len_cvs = len(cvs)


In [ ]:
# Plot of the free energy profiles of the selected structures as a function of the diffusion CV.
# Here we compare several local structure realizations at a single low pressure and a fixed gate size.
#
# The quantity shown is the relative free energy along the diffusion coordinate. The minimum is
# shifted to zero for clarity, which makes it easier to compare barrier shapes across structures.

base_workdir = 'OutputFiles/FlexibleDiffusionPath/ZIF8/C2H6_UA/AMBER_TRAPPE/atWBII_MFA_WDAc/1_0/3.7350/'

chempots = program.eos.compute_chempot(temperature=temperature, pressure=pressures)
chempot = chempots[0]

fig, ax = plt.subplots(1,1, figsize=(6,6))

for selection in range(5):
    fn = base_workdir + str(selection) + '/projected_density_energy.hdf5'

    with h5py.File(fn, 'r') as f:
        chempot_name = get_chempot_key(chempot)
        grp_name = f'T_{temperature:0.5f}/mu_{chempot_name}'
        grp = f[grp_name]
        
        q = grp['q'][:]
        free_energy = grp['free_energy'][:]
        rel_free_energy = (free_energy - np.nanmin(free_energy))/kjmol

        ax.plot(q/angstrom, rel_free_energy, label=f'{selection}')

ax.legend(title='Structure number')
ax.set_xlabel(r'CV $\AA$')
ax.set_ylabel('Free energy (kJ/mol)')
plt.show()


## Density averaging and free-energy construction

The density-averaging method is used to reduce the noise associated with individual local structures. For each gate size, the projected density is averaged over the available configurations before the corresponding free-energy profile is constructed.

This is useful because the admission barrier depends on the local opening of the framework, and each structure contributes a slightly different potential landscape. Averaging the densities gives a single representative profile for a nominal gate size while still keeping the underlying flexibility information in the ensemble of structures.

The free-energy profile is obtained from the projected density via the usual thermodynamic relation. The profile is then shifted so that the minimum is at zero to highlight barrier differences between gate sizes and pressures.


In [ ]:
# Calculating the density-averaged profile for each gate size.
# This step uses a single representative host structure for the gate size and then
# averages the projected densities over the full set of pressures and work directories.
#
# This is the density-averaging approach discussed in the previous markdown cell.
#
# NOTE: this cell can take several minutes to complete depending on the machine.

for gate_size in gate_sizes:
    hostname = 'ZIF8'
    
    hostchk = f'InputFiles/Host/ZIF8/ZIF8_Gates/{gate_size}/structure_0.pdb'
    hostpar = 'InputFiles/Host/ZIF8/AMBER_system.xml'
    
    host = NanoporousHost(hostname, hostchk, hostpar,
                    unit_energy='kjmol', unit_distance='angstrom', unit_sigma='nanometer', unit_mass='amu') #it is important to specify the units of the input file

    suffix = f'{gate_size}/density_average' # the average workdir must be placed in the same base directory

    # program initialization
    program = Program(prefix=prefix, hostname=hostname, guestname=guestname, 
                        ff_suffix=ff_suffix, funct_suffix=funct_suffix, grid_suffix=grid_suffix, suffix=suffix)

    program.set_system(host, guest)
    program.set_grid(npoints=[16,16,16])
    program.set_eos('MBWR')
    program.init_free_energy(temperature)

    program.set_temperature(temperature)

    calc = Calculator(program)
    calc.average_projected_density(temperature, pressures=pressures)
    for P in pressures:
        calc.free_energy_path(temperature=temperature, pressure=P, list_pressures=pressures)        



In [ ]:
# Plot the density-averaged free-energy profiles across gate sizes.
# The reference free energy is shifted to zero at its minimum for each curve so the
# barrier heights can be compared directly.

base_workdir = 'OutputFiles/FlexibleDiffusionPath/ZIF8/C2H6_UA/AMBER_TRAPPE/atWBII_MFA_WDAc/1_0/'

chempots = program.eos.compute_chempot(temperature=temperature, pressure=pressures)
chempot = chempots[0]

fig, ax = plt.subplots(1,1, figsize=(6,6))

for gs in gate_sizes:
    fn = base_workdir + gs + '/density_average/projected_density_energy.hdf5'

    with h5py.File(fn, 'r') as f:
        chempot_name = get_chempot_key(chempot)
        grp_name = f'T_{temperature:0.5f}/mu_{chempot_name}'
        grp = f[grp_name]
        
        q = grp['q'][:]
        free_energy = grp['free_energy'][:]
        rel_free_energy = free_energy - np.nanmin(free_energy)

        ax.plot(q / angstrom, rel_free_energy / kjmol, label=f'{gs} ' + r'$\AA$')

ax.legend(title='Gate size')
ax.set_xlabel(r'CV $\AA$')
ax.set_ylabel('Free energy (kJ/mol)')
plt.show()


## Flexibility integration strategy

Once the density-averaged profiles are available for different gate sizes, we combine them with the host flexibility free-energy profile. This integration step accounts for the fact that the framework does not remain at one fixed aperture but samples a distribution of flexible states.

The resulting profile is a better representation of the effective diffusion barrier in a flexible material than a single static barrier because it includes both:

- the diffusion barrier associated with each gate-opening state, and
- the thermodynamic weight of the host flexibility itself.

In practice, the flexibility contribution enters as a correction to the diffusion free energy, and the final curve can be compared across pressures or across different guest and host models.


In [ ]:
from cmmdft.integrate_flexibility import FreeEnergyProfile


In [ ]:
# Read the host flexibility FEP, which gives the free-energy cost associated with changing
# the gate size or a flexibility coordinate in the host framework.
fep_fn = 'InputFiles/Host/ZIF8/flexibility_fep.txt'
FEP = FreeEnergyProfile.from_txt(fep_fn, cvcol=0, fcol=1, cv_input_unit='au', f_input_unit='kjmol', delimiter=' ')

# Build the list of directories containing the averaged density profiles for each gate size.
base_dr = f'{prefix}/{hostname}/{guestname}/{ff_suffix}/{funct_suffix}/{grid_suffix}'
dr_list = [base_dr + f'/{gs}/density_average' for gs in gate_sizes]


In [ ]:
# This integration step combines the gate-size-dependent diffusion profiles with the host
# flexibility contribution to produce a single representative free-energy landscape.
flex_program = program.copy()
flex_program.alter_workdir(suffix='density_average')

flex_calc = Calculator(flex_program)
flex_calc.flexibility_integration(FEP, dr_list, gate_sizes_float*angstrom, temperature, pressures=pressures)


In [ ]:
# Show profiles as a function of pressure.
# This panel illustrates how the barrier shape changes with increasing pressure while the
# same host model and diffusion CV are used.

base_workdir = 'OutputFiles/FlexibleDiffusionPath/ZIF8/C2H6_UA/AMBER_TRAPPE/atWBII_MFA_WDAc/1_0/'
chempots = program.eos.compute_chempot(temperature=temperature, pressure=pressures)

fig, ax = plt.subplots(1,1, figsize=(6,6))

fn = base_workdir +  '/density_average/projected_density_energy.hdf5'

with h5py.File(fn, 'r') as f:
    for e, chempot in enumerate(chempots):
        P = pressures[e]/bar
        chempot_name = get_chempot_key(chempot)
        grp_name = f'T_{temperature:0.5f}/mu_{chempot_name}'
        grp = f[grp_name]
        
        q = grp['q'][:]
        free_energy = grp['free_energy'][:]
        rel_free_energy = free_energy - np.nanmin(free_energy)

        ax.plot(q/angstrom, rel_free_energy/kjmol, label=f'{P}')

ax.legend(title='Pressure (bar)')
ax.set_xlabel(r'CV $\AA$')
ax.set_ylabel('Free energy (kJ/mol)')
plt.show()


## Effective average potential alternative

The effective-average-potential method is an alternative route to the same physical idea. Instead of averaging the projected densities after solving each structure separately, we construct a single host potential by averaging the external potentials from the selected local structures.

This creates a representative static host potential at each gate size. The guest then experiences this averaged potential in a standard equilibrium calculation. The method is computationally efficient and often useful for quick comparisons, but it does not encode the same distributional information as the explicit density-averaging and flexibility-integration workflow.

In short:

- density averaging keeps the structural ensemble in the density before constructing the free-energy profile;
- effective-potential averaging compresses the ensemble into one host potential before solving the guest density.

Both methods aim to mimic the effect of framework flexibility in a tractable way.


In [ ]:
prefix = 'OutputFiles/EffectiveFlexibleDiffusionPath'

# Guest definition is the same as in the previous example.
guestname = 'C2H6_UA'
mass, sigma, epsilon = LJ_pars_dict['C2H6']

guestchk = 'InputFiles/Guest/C2H6/struct_CA.chk'
guestpar = 'InputFiles/Guest/C2H6/pars_vdw_CA_trappe.txt'
guest = DualModelGuest(guestname, mass, sigma, epsilon, struct=guestchk, par=guestpar, m=1, ffname='TRAPPE-PCSAFT', hs_def='exp')

ff_suffix = 'AMBER_TRAPPE'
funct_suffix = 'atWBII_MFA_WDAc'

grid_suffix = '1_0' # A low resolution is chosen for the sake of efficiency in this example

# Thermodynamic properties.
temperature = 300*kelvin
pressures = np.linspace(0.1,50,5)*bar

# Each gate size is represented by a set of local structures. The effective average host
# potential combines these structures into a single host model.
hostname = 'ZIF8'
gate_sizes_float = np.arange(3.5750, 3.7550, 0.04)
gate_sizes = [f'{gs:0.4f}' for gs in gate_sizes_float]
selections = np.arange(5)


In [ ]:
# Calculating the equilibrium density for the different gate sizes using an effective
# average host potential. This approach approximates flexibility by averaging the external
# potential over multiple local structures at a fixed gate size.

for gate_size in gate_sizes:
    host_struct_list = [f'InputFiles/Host/ZIF8/ZIF8_Gates/{gate_size}/structure_{selection}.pdb' for selection in selections]
    host_par_list = ['InputFiles/Host/ZIF8/AMBER_system.xml' for _ in selections]
    
    # The host is defined as an EffectiveAverageHost, which constructs the potential as an
    # effective average over the selected structures.
    host = EffectiveAverageHost('ZIF8', host_struct_list, host_par_list, struct=host_struct_list[0], par=host_par_list[0],
                      unit_energy='kjmol', unit_distance='angstrom', unit_sigma='nanometer', unit_mass='amu') 
    suffix = f'{gate_size}'

    # program initialization
    program = Program(prefix=prefix, hostname='ZIF8', guestname=guestname, 
                        ff_suffix=ff_suffix, funct_suffix=funct_suffix, grid_suffix=grid_suffix, suffix=suffix)

    program.set_system(host, guest)
    program.set_grid(npoints=[16,16,16])
    program.set_eos('MBWR')
    program.init_free_energy(temperature)

    program.fener.add_external_potential(temperature)
    program.fener.add_hard_sphere()
    program.fener.add_mean_field()
    program.set_temperature(temperature)

    # Defining multiple convergence criteria to improve stability.
    criteria = ['riue', 'res']
    thresholds = [1e-4, 1e-6]

    solver = Anderson(program, criterion=criteria, threshold=thresholds)
    program.set_solver(solver)

    calc = Calculator(program)

    for P in pressures:
        program.solve(pressure=P)
        calc.save_loading_and_grand_potential(temperature=temperature, pressure=P) # Saves loading and grand potential per calculation to an npz file for later use (necessary for later integration)

    carbon_indices = np.array([171,191,175,187,179,183]) # indices of the atoms defining the gate opening
    step_dist = program.grid.spacings[0]/np.sqrt(3) # because the diffusion axis is the 111 diagonal for this unit cell (adjust as needed)
    distance_from_axis = 7*angstrom # necessary to keep the integration to the relevant pores (adjust as needed)
    cvs_limits = np.array([-7.5,7.5])*angstrom # defines the edges of the cv range

    cvs, cvs_mat, dist_mask = calc.collective_variable(ring_indices=carbon_indices, dist_from_axis=distance_from_axis,
                                                        step_dist=step_dist, cvs_limits=cvs_limits, supercell=True)

    for P in pressures:
        calc.project_density(cvs, cvs_mat, dist_mask, temperature=temperature, pressure=P)
        calc.free_energy_path(temperature=temperature, pressure=P)        

len_cvs = len(cvs)


# Flexibility integration

We perform the flexibility integration again, now on the range of FEPs obtained with the effective average potential approach

In [ ]:
# Flexibility averaging is now performed in the same way as in the previous example.
# This step combines the effective average host potential with the host flexibility free-energy
# profile to produce a final representative barrier.

# Read the host flexibility FEP
fep_fn = 'InputFiles/Host/ZIF8/flexibility_fep.txt'
FEP = FreeEnergyProfile.from_txt(fep_fn, cvcol=0, fcol=1, cv_input_unit='au', f_input_unit='kjmol', delimiter=' ')

# Build a list of work directories containing the averaged profiles for each gate size.
base_dr = f'{prefix}/{hostname}/{guestname}/{ff_suffix}/{funct_suffix}/{grid_suffix}'
dr_list = [base_dr + f'/{gs}' for gs in gate_sizes]
flex_eff_program = program.copy()
flex_eff_program.alter_workdir(suffix='')

flex_eff_calc = Calculator(flex_program)
flex_eff_calc.flexibility_integration(FEP, dr_list, gate_sizes_float*angstrom, temperature, pressures=pressures)


In [ ]:
flex_eff_plotter = Plotter(flex_eff_calc)

fig = flex_eff_plotter.plot_free_energy_profile(temperature, pressures=pressures)